# OPSD — Paper-Comparable Evaluation on Google Colab

Evaluation of trained OPSD LoRA adapters **vs the base model**, reusing the repo's own eval method (`eval/evaluate_math.py`: vLLM + LoRA, `\boxed{}` answer extraction, `math_verify` grading, `Avg@N`). It auto-discovers adapters and intermediate checkpoints saved under `opsd_outputs/` and prints a `Base vs trained` table in the same style as the project doc.

> **Paper-comparable settings.** This notebook is configured for the Qwen3-1.7B headline-style setup: full AIME24 (`NUM_PROBLEMS=None`), `VAL_N=12`, thinking enabled, and `MAX_NEW_TOKENS=38912`. Expect this to be expensive: roughly 30–50 min per checkpoint on strong GPUs, and much longer on T4.

> **Note on saved models.** A training run only yields a loadable adapter if it reached `save_steps` (or finished). Runs that stopped early have **no weights** to evaluate and will be reported as such below.

## 1. GPU + Google Drive

Set the runtime to a GPU (**Runtime → Change runtime type → GPU**; an A100/L4 is ideal but a T4 also works for this lightweight run).

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type and select a GPU."
)
print("GPU:", torch.cuda.get_device_name(0))

# Mount Google Drive so we can read your trained adapters and write results.
from google.colab import drive
drive.mount("/content/drive")

Tue Jun 16 06:46:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             45W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Locate the project code

We need `eval/evaluate_math.py` from the OPSD project (the same folder you trained from). This finds it on your Drive automatically.

In [ ]:
import os, glob, sys

# Adjust if your OPSD folder lives elsewhere on Drive.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/OPSD-main"

def find_project_dir(root="/content"):
    hits = glob.glob(os.path.join(root, "**", "opsd_train.py"), recursive=True)
    return os.path.dirname(hits[0]) if hits else None

if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "opsd_train.py")):
    PROJECT_DIR = DRIVE_PROJECT_DIR
else:
    PROJECT_DIR = find_project_dir()

assert PROJECT_DIR, "Could not locate the OPSD project (opsd_train.py). Put OPSD-main on your Drive."
EVAL_DIR = os.path.join(PROJECT_DIR, "eval")
assert os.path.exists(os.path.join(EVAL_DIR, "evaluate_math.py")), f"evaluate_math.py not found under {EVAL_DIR}"
print("PROJECT_DIR =", PROJECT_DIR)
print("EVAL_DIR    =", EVAL_DIR)

PROJECT_DIR = /content/drive/MyDrive/OPSD-main
EVAL_DIR    = /content/drive/MyDrive/OPSD-main/eval


## 3. Install dependencies (vLLM eval stack)

Same pinned stack as training, plus `vllm` (the evaluator runs on vLLM with LoRA). vLLM is installed after the base stack, then `transformers`/`trl` are re-pinned so vLLM's resolver can't downgrade them.

> If Colab prompts to **restart the runtime** after install, do it, then re-run from cell 1. The `starlette`/`gradio` version warnings are harmless here.

In [ ]:
%pip install -q \
    "transformers==4.57.1" \
    "trl==0.26.0" \
    "peft==0.17.1" \
    "accelerate==1.11.0" \
    "datasets==3.6.0" \
    "math-verify==0.8.0" \
    "einops==0.8.1"

%pip install -q "vllm==0.11.0"

# vLLM 0.11.0 pins torch==2.8.0, but Colab's in-place upgrade can leave a mix of
# torch files from two versions, which surfaces as internal import errors like
# `ImportError: cannot import name 'CUSTOM_KEY' from 'torch.ao.quantization'`.
# Force a clean, single-version reinstall of torch so every torch file matches.
%pip install -q --force-reinstall --no-deps "torch==2.8.0"

%pip install -q "transformers==4.57.1" "trl==0.26.0"

# torchvision (Colab's preinstalled build) no longer matches torch 2.8.0 and breaks
# `import transformers` (`operator torchvision::nms does not exist`). This text-only
# pipeline never uses torchvision, so remove it; transformers skips the import when absent.
%pip uninstall -y -q torchvision

print("\nDone. >>> RESTART THE RUNTIME NOW (Runtime > Restart session) <<<, then re-run from cell 1.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.2/517.2 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 4. Find your trained adapters

Scans `OUTPUT_DIR` for LoRA adapters (folders containing `adapter_model.safetensors`) and groups them by their base model (read from each `adapter_config.json`). Final adapters are used by default; set `INCLUDE_CHECKPOINTS = True` to also evaluate intermediate `checkpoint-*` dirs.

In [5]:
import json
from collections import defaultdict

# === Evaluate ONLY the thinking run requested ===
OUTPUT_DIR  = "/content/drive/MyDrive/opsd_outputs"  # where your training runs were saved
TARGET_BASE = "Qwen/Qwen3-1.7B"
TARGET_RUN  = "colab_a100_thinking_paper"
STEPS       = [25, 50, 75, 100]

def has_adapter(d):
    return (os.path.exists(os.path.join(d, "adapter_model.safetensors"))
            or os.path.exists(os.path.join(d, "adapter_model.bin")))

def read_base_from_config(d):
    cfg = os.path.join(d, "adapter_config.json")
    if not os.path.exists(cfg):
        return None
    with open(cfg) as f:
        return json.load(f).get("base_model_name_or_path")

# Build the exact adapter list: final + checkpoint-25/50/75/100.
TARGET_REL_DIRS = [TARGET_RUN] + [f"{TARGET_RUN}/checkpoint-{s}" for s in STEPS]

GROUPS = defaultdict(list)  # base_model -> [adapter_dir, ...]
found = []
missing = []
wrong_base = []

for rel in TARGET_REL_DIRS:
    d = os.path.join(OUTPUT_DIR, rel)
    if not os.path.isdir(d) or not has_adapter(d):
        missing.append(rel)
        continue
    base = read_base_from_config(d)
    if base != TARGET_BASE:
        wrong_base.append((rel, base))
        continue
    GROUPS[TARGET_BASE].append(d)
    found.append((TARGET_BASE, d))

print("=" * 70)
print("Adapters selected for evaluation:")
if found:
    for base, d in found:
        print(f"  base={base}  ->  {os.path.relpath(d, OUTPUT_DIR)}")
else:
    print("  (none found — did training reach save_steps / finish?)")

if missing:
    print("\nMissing adapters (not found / no adapter weights):")
    for rel in missing:
        print(f"  - {rel}")

if wrong_base:
    print("\nSkipped adapters with unexpected base_model_name_or_path:")
    for rel, base in wrong_base:
        print(f"  - {rel}  (base={base})")

print("=" * 70)

Adapters selected for evaluation:
  base=Qwen/Qwen3-1.7B  ->  colab_a100_thinking_paper
  base=Qwen/Qwen3-1.7B  ->  colab_a100_thinking_paper/checkpoint-25
  base=Qwen/Qwen3-1.7B  ->  colab_a100_thinking_paper/checkpoint-50
  base=Qwen/Qwen3-1.7B  ->  colab_a100_thinking_paper/checkpoint-75
  base=Qwen/Qwen3-1.7B  ->  colab_a100_thinking_paper/checkpoint-100


## 5. Paper-comparable eval config

Defaults now match the doc's Qwen3-1.7B headline evaluation style: full AIME24, `Avg@12`, thinking mode, and long token budget.

- `NUM_PROBLEMS = None` means evaluate the **full dataset** (AIME24 has 30 problems).
- `VAL_N = 12` means 12 sampled solutions per problem (360 generations for AIME24).
- `ENABLE_THINKING = True` and `MAX_NEW_TOKENS = 38912` are expensive but align with the reported thinking-mode results.

In [6]:
# Evaluate both AIME24 and AIME25, thinking mode only.
DATASETS        = ["aime24", "aime25"]
NUM_PROBLEMS    = None        # None = full dataset (AIME24/AIME25 have 30 problems)
VAL_N           = 12          # paper uses Avg@12
MAX_NEW_TOKENS  = 38912       # paper thinking-mode eval budget
ENABLE_THINKING = True        # force thinking mode
TEMPERATURE     = 1.0         # doc eval temperature

problem_desc = "all" if NUM_PROBLEMS is None else NUM_PROBLEMS
print(f"datasets={DATASETS}  problems={problem_desc}  val_n={VAL_N}  max_new_tokens={MAX_NEW_TOKENS}  thinking={ENABLE_THINKING}")
if NUM_PROBLEMS is None:
    print(f"Total generations per model per dataset = 30 x {VAL_N} = {30 * VAL_N}")
else:
    print("Total generations per model per dataset will be printed after the dataset is loaded.")

datasets=['aime24', 'aime25']  problems=all  val_n=12  max_new_tokens=38912  thinking=True
Total generations per model per dataset = 30 x 12 = 360


## 6. Run evaluation (base vs each adapter)

For every base model found, this evaluates the **base** once and then **each adapter** on top of it, calling the repo's `evaluate_math.py` (so grading matches the doc exactly). Output streams live; each model reloads vLLM (`enforce_eager=True`, so no CUDA-graph capture).

In [7]:
import subprocess, shlex, time

RESULTS_DIR = os.path.join(OUTPUT_DIR, "eval_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = []  # one row per (dataset, base, adapter)

# Force thinking-mode eval for everything in this notebook run.
ENABLE_THINKING = True

def _run_one(dataset, base_model, checkpoint_dir, out_json):
    cmd = f"""{sys.executable} -u evaluate_math.py \
      --base_model {base_model} \
      --dataset {dataset} \
      --val_n {VAL_N} \
      --max_new_tokens {MAX_NEW_TOKENS} \
      --max_model_len {MAX_NEW_TOKENS + 1024} \
      --temperature {TEMPERATURE} \
      --tensor_parallel_size 1 \
      --gpu_memory_utilization 0.85 \
      --output_file {shlex.quote(out_json)}"""
    if NUM_PROBLEMS is not None:
        cmd += f" --num_samples {NUM_PROBLEMS}"
    if not ENABLE_THINKING:
        cmd += " --no_thinking"
    if checkpoint_dir is not None:
        cmd += f" --checkpoint_dir {shlex.quote(checkpoint_dir)}"

    env = dict(os.environ)
    env.update({"TOKENIZERS_PARALLELISM": "false", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
    proc = subprocess.Popen(
        shlex.split(cmd),
        cwd=EVAL_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0 or not os.path.exists(out_json):
        return None
    with open(out_json) as f:
        return json.load(f).get("average_at_n_pct")

if not GROUPS:
    print("No adapters selected. Re-run cell 4 (check OUTPUT_DIR), then this cell.")

# Evaluate ONLY the selected adapters on both datasets (no separate base eval).
t0 = time.time()
for dataset in DATASETS:
    for base, adapters in GROUPS.items():
        safe_base = base.replace("/", "_")
        print(f"\n{'#' * 70}\n# DATASET: {dataset} | BASE MODEL: {base}\n{'#' * 70}")

        for d in adapters:
            rel = os.path.relpath(d, OUTPUT_DIR)
            print(f"\n>>> [{dataset}] adapter: {rel}")
            tag = rel.replace("/", "_")
            out_json = os.path.join(RESULTS_DIR, f"{safe_base}__{tag}__{dataset}.json")
            acc = _run_one(dataset, base, d, out_json)
            RESULTS.append({"dataset": dataset, "base": base, "model": rel, "avg_at_n": acc, "out_json": out_json})

print(f"\nAll evaluations finished in {(time.time() - t0) / 60:.1f} min.")

# Minimal summary.
for r in RESULTS:
    print(f"{r['dataset']:<6} | {r['model']:<55} | Avg@{VAL_N}={r['avg_at_n']}")



######################################################################
# DATASET: aime24 | BASE MODEL: Qwen/Qwen3-1.7B
######################################################################

>>> [aime24] adapter: colab_a100_thinking_paper
INFO 06-16 06:53:24 [__init__.py:216] Automatically detected platform cuda.
2026-06-16 06:53:24.705929: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-16 06:53:24.737921: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Auto-setting top_p to 0.95 for thinki

KeyboardInterrupt: 

## 7. Results table

`Base vs trained` comparison in the same shape as the project doc's tables. Re-runnable without re-evaluating (it just renders `RESULTS`).

In [ ]:
col = f"Avg@{VAL_N}"
problem_desc = "all" if NUM_PROBLEMS is None else NUM_PROBLEMS
print(f"Dataset: {DATASET}  |  problems: {problem_desc}  |  thinking: {ENABLE_THINKING}  |  max_new_tokens: {MAX_NEW_TOKENS}\n")
print(f"{'Base model':<22} {'Model':<42} {col:>10}")
print("-" * 76)
for r in RESULTS:
    acc = "n/a" if r["avg_at_n"] is None else f"{r['avg_at_n']:.1f}%"
    print(f"{r['base']:<22} {r['model']:<42} {acc:>10}")

# Save a compact summary next to the per-model JSONs.
summary_path = os.path.join(RESULTS_DIR, f"summary_{DATASET}_valn{VAL_N}.json")
with open(summary_path, "w") as f:
    json.dump({"dataset": DATASET, "num_problems": NUM_PROBLEMS, "val_n": VAL_N,
               "max_new_tokens": MAX_NEW_TOKENS, "enable_thinking": ENABLE_THINKING,
               "include_checkpoints": INCLUDE_CHECKPOINTS,
               "results": RESULTS}, f, indent=2)
print(f"\nSaved summary to: {summary_path}")

## Notes & next steps

- **Current setting is paper-comparable:** `ENABLE_THINKING=True`, `VAL_N=12`, `MAX_NEW_TOKENS=38912`, full dataset (`NUM_PROBLEMS=None`), and `INCLUDE_CHECKPOINTS=True` for per-step tables.
- **Runtime:** expect ~30–50 min **per checkpoint** on strong GPUs; much longer on T4. Evaluating base + checkpoints 25/50/75/100 can take several hours.
- **Missing adapters:** any run flagged `[NO SAVED ADAPTER]` never reached `save_steps`. Re-run training so it saves checkpoints at 25/50/75/100, then re-run this notebook — they will be picked up automatically.
- **Quick debugging:** temporarily set `NUM_PROBLEMS=3`, `VAL_N=1`, and `MAX_NEW_TOKENS=1024`, then restore the paper settings for the final run.